# Import Libraries

In [179]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xg

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Data Collection

In [180]:
train_df = pd.read_csv(r'train.csv')
test_df = pd.read_csv(r'test.csv')

## Concatenation of DataFrames

In [181]:
house_df = pd.concat([train_df, test_df], axis=0)

# Exploratory Data Analysis (EDA)

In [182]:
def learn_df(df, name):
    """
    Function to learn about a DataFrame.
    """
    print(f"DataFrame: {name}")
    print("_______________________________________________________________")
    print(f"Sample Data:\n{df.head()}")
    print("_______________________________________________________________")
    print(f"Description:\n{df.describe(include='all')}")
    print("_______________________________________________________________")
    print(f"Info:\n{df.info()}")
    print("_______________________________________________________________")
    print(f"Missing Values:\n{df.isnull().sum()}")
    print("_______________________________________________________________")
    print(f"Unique Values:\n{df.nunique()}")
    print("_______________________________________________________________")
    print(f"Duplicates:\n{df.duplicated().sum()}")
    print("_______________________________________________________________")
    
learn_df(house_df, "All House Dataset")

DataFrame: All House Dataset
_______________________________________________________________
Sample Data:
   ID                            fullAddress postcode  country outcode  \
0   0       38 Adelina Grove, London, E1 3AD   E1 3AD  England      E1   
1   1      6 Cleveland Grove, London, E1 4XL   E1 4XL  England      E1   
2   2   65 Sanderstead Road, London, E10 7PW  E10 7PW  England     E10   
3   3  5 Queenswood Gardens, London, E11 3SE  E11 3SE  England     E11   
4   4     12 Woodlands Road, London, E11 4RW  E11 4RW  England     E11   

    latitude  longitude  bathrooms  bedrooms  floorAreaSqM  livingRooms  \
0  51.519406  -0.053261        NaN       3.0          80.0          1.0   
1  51.521261  -0.053384        2.0       4.0         110.0          1.0   
2  51.569054  -0.034892        1.0       3.0          84.0          1.0   
3  51.564212   0.026292        NaN       2.0          72.0          1.0   
4  51.563430   0.006260        1.0       3.0         104.0          1.0   

# Data Cleaning / Data Preprocessing

## Log Transformation

In [183]:
house_df[['price', 'floorAreaSqM']] = np.log(house_df[['price', 'floorAreaSqM']])

## Splitting Full Address

In [184]:
house_df[['street', 'city', 'postcode']] = house_df['fullAddress'].str.rsplit(', ', expand = True, n = 2)

## Filling NaN Values for floorAreaSqM, bathrooms, bedrooms, and livingRooms

In [185]:
median_floor = house_df.groupby(['country', 'outcode'])['floorAreaSqM'].transform('median').round()
house_df['floorAreaSqM'] = house_df['floorAreaSqM'].fillna(median_floor)

median_bath = house_df.groupby(['floorAreaSqM'])['bathrooms'].transform('median').round()
median_house_bath = house_df.groupby(['country'])['bathrooms'].transform('median').round()
median_bath = median_bath.fillna(median_house_bath)
house_df['bathrooms'] = house_df['bathrooms'].fillna(median_bath)

median_bed = house_df.groupby(['floorAreaSqM'])['bedrooms'].transform('median').round()
median_house_bed = house_df.groupby(['country'])['bedrooms'].transform('median').round()
median_bed = median_bed.fillna(median_house_bed)
house_df['bedrooms'] = house_df['bedrooms'].fillna(median_bed)

median_livi = house_df.groupby(['floorAreaSqM'])['livingRooms'].transform('median').round()
median_house_livi = house_df.groupby(['country'])['livingRooms'].transform('median').round()
median_livi = median_livi.fillna(median_house_livi)
house_df['livingRooms'] = house_df['livingRooms'].fillna(median_livi)

## Feature Mapping for Regions

In [186]:
regions_map = {
    'Leytonstone':1, 'Walthamstow':1, 'Leyton':1, 'Stratford':1, 'Chingford':1,
    'Forest Gate':1, 'Woodford Green':1, # East London
    'London':2, # Central London
    'Southgate':3, 'Wembley':3,'Edmonton':3, 'Palmers Green':3, # North London
    'Blackheath':4, 'Woolwich':4, 'Kidbrooke':4, 'Charlton':4, 'Abbey Wood':4,
    'Greenwich':4, 'Eltham':4,  'Deptford':4, # South East London
    'Wimbledon':5, 'Raynes Park':5, 'Colliers Wood':5, # South West London
    'Acton':6, 'West Ealing':6, 'Ealing':6, 'Hanwell':6, 'Chiswick':6, 'Park Royal':6, # West London
    'Plumstead':7,'Bromley':7 # South London
    }
house_df['regions'] = house_df['city'].map(regions_map)

## Feature Mapping from more Expensive Outcodes to Less

In [187]:
outcode_map = {
    'W1B': 4, 'W1C': 4, 'W1D': 4, 'W1F': 4, 'W1G': 4, 'W1H': 4, 'W1J': 4, 'W1K': 4, 'W1S': 4, 'W1T': 4,
    'W1U': 4, 'W1W': 4, 'SW1A': 4, 'SW1E': 4, 'SW1H': 4, 'SW1P': 4, 'SW1V': 4, 'SW1W': 4, 'SW1X': 4, 'SW1Y': 4,
    'WC1A': 4, 'WC1B': 4, 'WC1E': 4, 'WC1H': 4, 'WC1N': 4, 'WC1R': 4, 'WC1V': 4, 'WC1X': 4,'WC2A': 4, 'WC2B': 4,
    'WC2E': 4, 'WC2H': 4, 'WC2N': 4, 'WC2R': 4, 'W10':4, # Most Expensive (Prime Central London)
    'EC1A': 3, 'EC1M': 3, 'EC1N': 3, 'EC1R': 3, 'EC1V': 3, 'EC1Y': 3, 'EC2A': 3, 'EC2M': 3, 'EC2N': 3, 'EC2R': 3,
    'EC2V': 3, 'EC2Y': 3, 'EC3A': 3, 'EC3M': 3, 'EC3N': 3, 'EC3R': 3, 'EC3V': 3, 'EC4A': 3, 'EC4M': 3, 'EC4R': 3,
    'EC4V': 3, 'EC4Y': 3, 'SW3': 3, 'SW5': 3, 'SW6': 3, 'SW7': 3, 'SW10': 3, 'SW11': 3, 'W2': 3, 'W8': 3,
    'W9': 3, 'W11': 3, 'W14': 3, # Expensive (Central London and Prime Areas)   
    'N1': 2, 'N2': 2, 'N3': 2, 'N4': 2, 'N5': 2, 'N6': 2, 'N7': 2, 'N8': 2, 'N10': 2, 'N11': 2,
    'N12': 2, 'N13': 2, 'N14': 2, 'N15': 2, 'N16': 2, 'N19': 2, 'N20': 2, 'N21': 2, 'N22': 2,'NW1': 2,
    'NW2': 2, 'NW3': 2, 'NW5': 2, 'NW6': 2, 'NW8': 2, 'NW10': 2, 'NW11': 2, 'SE1': 2, 'SE10': 2, 'SE11': 2,
    'SE15': 2, 'SE16': 2, 'SE21': 2, 'SE22': 2, 'SE24': 2, 'SW2': 2, 'SW4': 2, 'SW8': 2, 'SW9': 2, 'SW12': 2,
    'SW13': 2, 'SW14': 2, 'SW15': 2, 'SW16': 2, 'SW17': 2, 'SW18': 2, 'SW19': 2, 'SW20': 2, 'W3': 2, 'W4': 2,
    'W5': 2, 'W6': 2, 'W7': 2, 'W12': 2, 'W13': 2, # Moderately Expensive (Outer Central London and Suburban Prime Areas)
    'E1': 1, 'E2': 1, 'E3': 1, 'E4': 1, 'E5': 1, 'E6': 1, 'E7': 1, 'E8': 1, 'E9': 1, 'E10': 1,
    'E11': 1, 'E12': 1, 'E13': 1, 'E14': 1, 'E15': 1, 'E16': 1, 'E17': 1, 'E18': 1, 'E1W': 1,'IG8': 1,
    'N9': 1, 'N17': 1, 'N18': 1, 'NW4': 1, 'NW7': 1, 'NW9': 1,'SE2': 1, 'SE3': 1, 'SE4': 1, 'SE5': 1,
    'SE6': 1, 'SE7': 1, 'SE8': 1, 'SE9': 1, 'SE12': 1, 'SE13': 1, 'SE14': 1, 'SE17': 1, 'SE18': 1, 'SE19': 1,
    'SE20': 1, 'SE23': 1, 'SE25': 1, 'SE26': 1, 'SE27': 1, 'SE28': 1 # Less Expensive (Outer London and Suburban Areas)
    }
house_df['expen_outcode'] = house_df['outcode'].map(outcode_map)

## Handling Categorical Features

In [188]:
house_df['city'] = house_df['city']
house_df = pd.get_dummies(house_df, columns=['city'])

house_df['currentEnergyRating'] = house_df['currentEnergyRating'].fillna('Unknown')
house_df = pd.get_dummies(house_df, columns=['currentEnergyRating'])

house_df['tenure'] = house_df['tenure'].fillna('Unknown')
house_df = pd.get_dummies(house_df, columns=['tenure'])

house_df['propertyType'] = house_df['propertyType'].fillna('Unknown')
house_df = pd.get_dummies(house_df, columns=['propertyType'])

season_map = {1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 3, 7: 3, 8: 3, 9: 4, 10: 4, 11: 4, 12: 1}
house_df['seasons'] = house_df['sale_month'].map(season_map)

## Drop Unessacery Columns

In [189]:
house_df = house_df.drop(columns = ['fullAddress', 'postcode', 'country', 'street', 'outcode'], axis = 1)

In [190]:
print(house_df.columns.to_list())

['ID', 'latitude', 'longitude', 'bathrooms', 'bedrooms', 'floorAreaSqM', 'livingRooms', 'sale_month', 'sale_year', 'price', 'regions', 'expen_outcode', 'city_Abbey Wood', 'city_Acton', 'city_Blackheath', 'city_Bromley', 'city_Charlton', 'city_Chingford', 'city_Chiswick', 'city_Colliers Wood', 'city_Deptford', 'city_Ealing', 'city_Edmonton', 'city_Eltham', 'city_Forest Gate', 'city_Greenwich', 'city_Hanwell', 'city_Kidbrooke', 'city_Leyton', 'city_Leytonstone', 'city_London', 'city_Palmers Green', 'city_Park Royal', 'city_Plumstead', 'city_Raynes Park', 'city_Southgate', 'city_Stratford', 'city_Walthamstow', 'city_Wembley', 'city_West Ealing', 'city_Wimbledon', 'city_Woodford Green', 'city_Woolwich', 'currentEnergyRating_A', 'currentEnergyRating_B', 'currentEnergyRating_C', 'currentEnergyRating_D', 'currentEnergyRating_E', 'currentEnergyRating_F', 'currentEnergyRating_G', 'currentEnergyRating_Unknown', 'tenure_Feudal', 'tenure_Freehold', 'tenure_Leasehold', 'tenure_Shared', 'tenure_Unkn

## Split house_df back into train_df and test_df

In [191]:
train_df = house_df.iloc[:len(train_df)].reset_index(drop=True)
test_df = house_df.iloc[len(train_df):len(train_df) + len(test_df)].reset_index(drop=True)

features = [
    'latitude', 'longitude', 'bathrooms', 'bedrooms', 'floorAreaSqM',
    'livingRooms', 'sale_month', 'sale_year', 'regions', 'expen_outcode',
    'city_Abbey Wood', 'city_Acton', 'city_Blackheath', 'city_Bromley', 'city_Charlton',
    'city_Chingford', 'city_Chiswick', 'city_Colliers Wood', 'city_Deptford', 'city_Ealing',
    'city_Edmonton', 'city_Eltham', 'city_Forest Gate', 'city_Greenwich', 'city_Hanwell',
    'city_Kidbrooke', 'city_Leyton', 'city_Leytonstone', 'city_London', 'city_Palmers Green',
    'city_Park Royal', 'city_Plumstead', 'city_Raynes Park', 'city_Southgate', 'city_Stratford',
    'city_Walthamstow', 'city_Wembley', 'city_West Ealing', 'city_Wimbledon', 'city_Woodford Green',
    'city_Woolwich', 'currentEnergyRating_A', 'currentEnergyRating_B', 'currentEnergyRating_C',
    'currentEnergyRating_D', 'currentEnergyRating_E', 'currentEnergyRating_F', 'currentEnergyRating_G',
    'currentEnergyRating_Unknown', 'tenure_Feudal', 'tenure_Freehold', 'tenure_Leasehold',
    'tenure_Shared', 'tenure_Unknown', 'propertyType_Bungalow Property', 'propertyType_Converted Flat',
    'propertyType_Detached Bungalow', 'propertyType_Detached House', 'propertyType_Detached Property',
    'propertyType_End Terrace Bungalow', 'propertyType_End Terrace House', 'propertyType_End Terrace Property',
    'propertyType_Flat/Maisonette', 'propertyType_Mid Terrace Bungalow', 'propertyType_Mid Terrace House',
    'propertyType_Mid Terrace Property', 'propertyType_Purpose Built Flat', 'propertyType_Semi-Detached Bungalow',
    'propertyType_Semi-Detached House', 'propertyType_Semi-Detached Property', 'propertyType_Terrace Property',
    'propertyType_Terraced', 'propertyType_Terraced Bungalow', 'propertyType_Unknown', 'seasons'
]

# Modeling - Evaluation

In [192]:
X = train_df[features]
y = train_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [217]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_pred = np.exp(y_pred)
    y_test = np.exp(y_test)
    return {
        "R^2 Score":r2_score(y_test, y_pred),
        "Mean Absolute Error":mean_absolute_error(y_test, y_pred),
        "Mean Squared Error":mean_squared_error(y_test, y_pred),
        "Root Mean Squared Error":np.sqrt(mean_squared_error(y_test, y_pred))
    }

models = {
#     "Random Forest Regression": RandomForestRegressor(
#     n_estimators=80,
#     max_depth=10,
#     min_samples_split=20,
#     min_samples_leaf=5,
#     max_features='sqrt',
#     bootstrap=True,
#     oob_score=True,
#     random_state=42
# ),
    "XGBoost Regression": xg.XGBRegressor(
    n_estimators=3000, # Number of trees to fit
    objective = "reg:squarederror", # Objective function
    max_depth=11, # Maximum depth of a tree
    learning_rate=0.04,
    subsample=0.8, # Subsample ratio of the training instance
    colsample_bytree=0.8, # Subsample ratio of columns when constructing each tree
    gamma=0.05, # Minimum loss reduction required to make a further partition on a leaf node
    min_child_weight=3, # Minimum sum of instance weight (hessian) needed in a child
    reg_alpha=0.5, # L1 regylarization term on weights
    reg_lambda=1.0, # L2 regularization term on weights
    random_state=42
),
    # "Gradient Boosting Regression": GradientBoostingRegressor(
    # n_estimators=80,
    # learning_rate=0.08,
    # max_depth=20,
    # min_samples_split=10,
    # min_samples_leaf=5,
    # max_features='sqrt',
    # subsample=0.8,
    # loss='huber',
    # random_state=42
# )
}

## Train each model only once!
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)  # Train once!
    metrics = evaluate_model(model, X_test, y_test)
    results[name] = metrics
    print(f"Model : {name}")
    for metric, value in metrics.items():
        print(f"{metric} : {value:.4f}")
    print("_____________________________________________")

Model : XGBoost Regression
R^2 Score : 0.5673
Mean Absolute Error : 135891.1444
Mean Squared Error : 605526048898.1490
Root Mean Squared Error : 778155.5429
_____________________________________________


## Determine the best model based on MAE

In [218]:
best_model_name = min(results, key=lambda name: results[name]['Mean Absolute Error'])
best_model = models[best_model_name]
print(f"Best model selected: {best_model_name}")

Best model selected: XGBoost Regression


## Now, connect (or “chain”) your create_submission function with the best model.

In [219]:
def create_submission(best_model, test_df, features, id_col='ID', filename='London_Price_Predictions.csv'):
    submission_df = test_df.copy()
    submission_df['price'] = best_model.predict(submission_df[features])
    submission_df['price'] = np.exp(submission_df['price'])
    London_Price_Predictions = submission_df[[id_col, 'price']]
    London_Price_Predictions.to_csv(filename, index=False)
    print(f"Submission file saved as '{filename}'")

## Call create_submission with the best model and your test dataset

In [220]:
create_submission(best_model, test_df, features, id_col='ID', filename='London_Price_Predictions.csv')

Submission file saved as 'London_Price_Predictions.csv'
